In [2]:
from faster_whisper import WhisperModel

model = WhisperModel("base", compute_type="float16")

segments, _ = model.transcribe("/home/muthuajay/Documents/GitHub/STT/data/technical_audio_01.wav")

text = " ".join([seg.text for seg in segments])

print(text)

/home/muthuajay/Documents/GitHub/STT/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 In this experiment, we are initializing a faster R-C-N-N object detection model,  with a custom MBCO and V-Net backbone.  To accelerate the training loop for face detection,  we implement distributed data parallel, or DDP, across a multi-GPU setup in PyTorch.  It is crucial to correctly synchronize the gradients and manage the region proposal network,  and region of interest pooling layers.  We will also integrate a local LLM with the RAG.  Architecture for the text-to-sql retrieval pipeline.


In [3]:
import pandas as pd

transcripts_df = pd.read_csv("/home/muthuajay/Documents/Datasets/LJSpeech-1.1/metadata.csv", sep="|", header=None, names=["file_id", "transcript", "normalized_transcript"])

transcripts_df.head()

,file_id,transcript,normalized_transcript
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...


In [4]:
def transcribe_and_compare(file_id):
    audio_path = f"/home/muthuajay/Documents/Datasets/LJSpeech-1.1/wavs/{file_id}.wav"
    segments, _ = model.transcribe(audio_path)
    transcribed_text = " ".join([seg.text for seg in segments])
    
    original_transcript = transcripts_df[transcripts_df['file_id'] == file_id]['transcript'].values[0]
    
    print(f"Original Transcript: {original_transcript}")
    print(f"Transcribed Text: {transcribed_text}")


# Example usage
sample_df = transcripts_df.sample(5)
for file_id in sample_df['file_id']:
    transcribe_and_compare(file_id)

Original Transcript: and in one corner, at some depth, a bundle of clothes were unearthed, which, with a hairy cap,
Transcribed Text:  And in one corner, at some depth, a bundle of clothes were unearthed, which, with a hairy  cap...
Original Transcript: was stamped on some literature that Oswald had in his possession at the time of his arrest in New Orleans,
Transcribed Text:  was stamped on some literature that Oswald had in his possession at the time of his arrest  in New Orleans.
Original Transcript: Until the time of his death he kept a small shop close to the church in Horncastle.
Transcribed Text:  Until the time of his death, he kept a small shop close to the church in Horncastle.
Original Transcript: Efforts made by the Bureau since the assassination, on the other hand,
Transcribed Text:  Efforts made by the Bureau since the assassination, on the other hand.
Original Transcript: Owing to the vast size of the place, the inhabitants of the central parts (as the residents of Babyl

In [15]:
import sounddevice as sd
from scipy.io.wavfile import write

fs = 16000  # Whisper prefers 16kHz
seconds = 10

print("Recording...")
audio = sd.rec(int(seconds * fs), samplerate=fs, channels=1)
sd.wait()

write("pytorch.wav", fs, audio)
print("Saved!")

Recording...
Saved!


In [1]:
def transcribe( model: WhisperModel , audio_path):
    segments, _ = model.transcribe(audio_path)
    transcribed_text = " ".join([seg.text for seg in segments])
    return transcribed_text

transcribed_text = transcribe(model, "/home/muthuajay/Documents/GitHub/STT/data/technical_audio_01.wav")
print(transcribed_text)

NameError: name 'WhisperModel' is not defined

In [16]:
transcribed_text = transcribe(model, "pytorch.wav")
print(transcribed_text)

 Pytosh is one of the best open source model that is available and it is easier to use  this framework then tends to flow.


In [ ]:
We trained a transformer model using PyTorch and fine tuned it on a custom dataset for better accuracy.

Machine learning models learn patterns from data and improve their performance over time without being explicitly programmed.


Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data and improve their performance over time. Instead of being explicitly programmed, these models identify patterns and make predictions based on input data. Common techniques include supervised learning, unsupervised learning, and reinforcement learning. For example, in supervised learning, a model is trained on labeled data to predict outcomes accurately. Modern approaches such as deep learning and transformer architectures have significantly advanced tasks like image recognition and natural language processing, enabling applications such as chatbots, recommendation systems, and autonomous vehicles.

In [20]:
from ollama import chat

def lm_correct(text: str) -> str:
    prompt = f"""
You are a speech-to-text correction system.

Fix transcription errors in the sentence below.
- Correct technical terms (e.g., PyTorch, Transformer)
- Fix grammar and missing words
- Do NOT change meaning
- Keep it concise

Input:
{text}

Output:
"""

    response = chat(
        model='qwen3.5:4b',
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response['message']['content'].strip()

corrected_text = lm_correct(transcribed_text)
print("Corrected Text:", corrected_text)

Corrected Text: PyTorch is one of the best open-source frameworks available and it is easier to use this framework than others.
